# Agregações e Análise Estatística — YouTube com PySpark

Neste notebook exploramos os dados enriquecidos de vídeos do YouTube por meio de diversas operações de agregação: contagens, médias, variância, min/max, janelas temporais e média acumulativa.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('YouTube - Agregacoes') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession iniciada | Spark {spark.version}')

## Passo 1 — Leitura do arquivo `videos-preparados.snappy.parquet`

In [ ]:
# Passo 1: Leitura do arquivo parquet no dataframe df_video
df_video = spark.read.parquet('videos-preparados.snappy.parquet')

print('Esquema do df_video:')
df_video.printSchema()
print(f'Total de registros: {df_video.count()}')
df_video.show(5, truncate=True)

## Passo 2 — Quantidade de registros por `Keyword`

In [ ]:
# Passo 2: Quantidade de registros para cada valor único da coluna 'Keyword'
df_count_keyword = df_video \
    .groupBy('Keyword') \
    .count() \
    .orderBy('count', ascending=False)

print('Quantidade de registros por Keyword:')
df_count_keyword.show(41, truncate=False)

## Passo 3 — Média de `Interaction` por `Keyword`

In [ ]:
# Passo 3: Média da coluna 'Interaction' para cada valor único da coluna 'Keyword'
df_avg_interaction = df_video \
    .groupBy('Keyword') \
    .agg(F.avg('Interaction').alias('Avg Interaction')) \
    .orderBy('Avg Interaction', ascending=False)

print('Média de Interaction por Keyword:')
df_avg_interaction.show(41, truncate=False)

## Passo 4 — Máximo de `Interaction` por `Keyword` → `Rank Interactions` (ordem decrescente)

In [ ]:
# Passo 4: Valor máximo de 'Interaction' por 'Keyword', nomeado 'Rank Interactions',
# ordenado de forma decrescente
df_rank_interactions = df_video \
    .groupBy('Keyword') \
    .agg(F.max('Interaction').alias('Rank Interactions')) \
    .orderBy('Rank Interactions', ascending=False)

print('Rank Interactions (máximo de Interaction por Keyword):')
df_rank_interactions.show(41, truncate=False)

## Passo 5 — Média e Variância de `Views` por `Keyword`

In [ ]:
# Passo 5: Média e variância da coluna 'Views' para cada valor único de 'Keyword'
df_views_stats = df_video \
    .groupBy('Keyword') \
    .agg(
        F.avg('Views').alias('Avg Views'),
        F.variance('Views').alias('Variance Views')
    ) \
    .orderBy('Avg Views', ascending=False)

print('Média e Variância de Views por Keyword:')
df_views_stats.show(41, truncate=False)

## Passo 6 — Média, Mínimo e Máximo de `Views` por `Keyword` (sem casas decimais)

In [ ]:
# Passo 6: Média, mínimo e máximo de 'Views' por 'Keyword', sem casas decimais
df_views_minmax = df_video \
    .groupBy('Keyword') \
    .agg(
        F.round(F.avg('Views'), 0).cast('long').alias('Avg Views'),
        F.min('Views').alias('Min Views'),
        F.max('Views').alias('Max Views')
    ) \
    .orderBy('Avg Views', ascending=False)

print('Média, Mínimo e Máximo de Views por Keyword (sem casas decimais):')
df_views_minmax.show(41, truncate=False)

## Passo 7 — Primeiro e Último `Published At` por `Keyword`

In [ ]:
# Passo 7: Primeiro e último 'Published At' para cada valor único da coluna 'Keyword'
df_published_range = df_video \
    .groupBy('Keyword') \
    .agg(
        F.min('Published At').alias('Primeiro Published At'),
        F.max('Published At').alias('Ultimo Published At')
    ) \
    .orderBy('Keyword')

print('Primeiro e Último Published At por Keyword:')
df_published_range.show(41, truncate=False)

## Passo 8 — Contagem total e única de `Title`

In [ ]:
# Passo 8: Contar todos os 'Title' e todos os únicos, verificando se há diferença
total_titles  = df_video.count()
unique_titles = df_video.select('Title').distinct().count()
diferenca     = total_titles - unique_titles

print(f'Total de Title (incluindo duplicatas): {total_titles}')
print(f'Total de Title únicos:                 {unique_titles}')
print(f'Diferença (títulos duplicados):        {diferenca}')

if diferenca > 0:
    print(f'\n⚠ Existem {diferenca} títulos duplicados no dataset.')
    print('\nExemplos de títulos duplicados:')
    df_video.groupBy('Title') \
        .count() \
        .filter(F.col('count') > 1) \
        .orderBy('count', ascending=False) \
        .show(10, truncate=True)
else:
    print('\n✓ Não há títulos duplicados no dataset.')

## Passo 9 — Quantidade de registros por Ano (ordem ascendente)

In [ ]:
# Passo 9: Quantidade de registros ordenados por ano em ordem ascendente
df_count_year = df_video \
    .groupBy('Year') \
    .count() \
    .orderBy('Year', ascending=True)

print('Quantidade de registros por Ano (ordem ascendente):')
df_count_year.show(50, truncate=False)

## Passo 10 — Quantidade de registros por Ano e Mês (ordem ascendente)

In [ ]:
# Passo 10: Quantidade de registros ordenados por ano e mês em ordem ascendente
df_count_year_month = df_video \
    .groupBy('Year', 'Month') \
    .count() \
    .orderBy('Year', 'Month', ascending=True)

print('Quantidade de registros por Ano e Mês (ordem ascendente):')
df_count_year_month.show(200, truncate=False)

## Passo 11 — Média Acumulativa de `Likes` por `Keyword` ao longo dos anos

In [ ]:
# Passo 11: Média acumulativa de 'Likes' para cada 'Keyword' ao longo dos anos
# Passo 11a: Calcular a média anual de Likes por Keyword
df_avg_likes_year = df_video \
    .groupBy('Keyword', 'Year') \
    .agg(F.avg('Likes').alias('Avg Likes Year'))

# Passo 11b: Definir a janela particionada por Keyword e ordenada por Year
# A média acumulativa usa ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
window_cumulative = Window \
    .partitionBy('Keyword') \
    .orderBy('Year') \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Passo 11c: Aplicar a média acumulativa
df_cumulative_avg_likes = df_avg_likes_year \
    .withColumn(
        'Cumulative Avg Likes',
        F.round(F.avg('Avg Likes Year').over(window_cumulative), 2)
    ) \
    .orderBy('Keyword', 'Year')

print('Média Acumulativa de Likes por Keyword ao longo dos Anos:')
df_cumulative_avg_likes.show(100, truncate=False)

In [ ]:
# Encerramento da SparkSession
spark.stop()
print('SparkSession encerrada com sucesso!')